# 11. Validation humaine vs LLM

Ce notebook exploite le protocole de validation humaine (tâche A1) : échantillon de 150 textes triplement stratifié (bloc × arène × batch), annotations manuelles, et comparaison avec l'annotation LLM (stance_v3). On calcule Cohen's Kappa, Spearman, accord exact/±1, biais par bloc, matrice de confusion, et on produit les figures fig51 et fig52.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
import matplotlib.pyplot as plt

from config import FIGURES_DIR
from validation_metrics import load_annotated_sample, compute_metrics, plot_figures, write_report

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

## Chargement et métriques

Le sample est chargé depuis `data/validation/sample_150.csv` et `annotations.csv` (ou colonne stance_humain). Merge avec le corpus pour récupérer stance_v3.

In [ ]:
df, err = load_annotated_sample()
if err:
    print(err)
else:
    results = compute_metrics(df)
    print("Cohen kappa =", results.get("kappa", "N/A"))
    print("Spearman rho =", results.get("spearman_rho", "N/A"))
    print("Accord exact =", results.get("accord_exact", 0), "%")
    print("Accord ±1 =", results.get("accord_pm1", 0), "%")
    print("Biais par bloc:", results.get("bias_by_bloc", {}))

## Figures fig51 et fig52

In [ ]:
if not err:
    plot_figures(results, figures_dir=FIG_DIR)

## Analyse de sensibilité

Si on corrige le biais systématique détecté (soustraire mean(LLM - humain) par bloc au score LLM), est-ce que les résultats principaux R2 (paradoxe Droite au cessez-le-feu) et R4 (polarisation lexicale) survivent ?

Cette analyse nécessite de rejouer les event studies et la polarisation sur le corpus avec stance corrigé. Ici on illustre le principe : si le biais est faible et homogène, la correction a peu d'effet.

In [ ]:
if not err and results.get("bias_by_bloc"):
    biases = results["bias_by_bloc"]
    max_bias = max(abs(b) for b in biases.values()) if biases else 0
    print(f"Biais max par bloc : {max_bias:.3f}")
    if max_bias < 0.3:
        print("Conclusion : biais faible ; R2 et R4 probablement robustes.")
    else:
        print("Conclusion : analyser l'impact d'une correction du biais sur R2 et R4.")